# Sprint 5 — Testing & Deployment
## QA, End-to-End Testing & Production Deployment
Validates all Jira user stories, runs a full simulated match end-to-end, and deploys the backend + dashboard.

In [1]:
import requests, json, time
print("Sprint 5 — Testing & Deployment")
print("Ensure API is running: uvicorn app:app --reload --port 8000")
BASE_URL = "http://localhost:8000"
TEAM_A = ["Rohit Sharma","Ishan Kishan","Suryakumar Yadav","Hardik Pandya",
          "Kieron Pollard","Krunal Pandya","Dwayne Bravo","Jasprit Bumrah",
          "Trent Boult","Lasith Malinga","Mitchell McClenaghan"]
TEAM_B = ["Ruturaj Gaikwad","Devon Conway","Faf du Plessis","Ambati Rayudu",
          "MS Dhoni","Ravindra Jadeja","Moeen Ali","Deepak Chahar",
          "Imran Tahir","Yuzvendra Chahal","Shardul Thakur"]

Sprint 5 — Testing & Deployment
Ensure API is running: uvicorn app:app --reload --port 8000


## 1. Jira User Story Acceptance Tests

In [2]:
passed = 0; failed = 0; results = []

def assert_test(name, condition, detail=""):
    global passed, failed
    status = "PASS" if condition else "FAIL"
    if condition: passed += 1
    else: failed += 1
    emoji = "OK" if condition else "XX"
    print(f"[{emoji}] {name}")
    if detail and not condition: print(f"     -> {detail}")

try:
    r = requests.post(f"{BASE_URL}/match/setup", json={
        "team_a": "MI", "team_b": "CSK",
        "team_a_players": TEAM_A, "team_b_players": TEAM_B,
        "batting_first": "A", "venue": "Wankhede Stadium"
    }, timeout=5)
    assert_test("US-01: Match setup returns 200", r.status_code == 200)
    d = r.json()
    assert_test("US-01: Setup has status key", "status" in d)

    overs_data = [
        {"over_number":1, "balls":["0","4","1","0","6","1"], "bowler":"Deepak Chahar"},
        {"over_number":2, "balls":["2","0","W","0","4","1"], "bowler":"Yuzvendra Chahal"},
        {"over_number":3, "balls":["1","6","0","4","W","0"], "bowler":"Imran Tahir"},
    ]
    last_state = None
    for ov in overs_data:
        r = requests.post(f"{BASE_URL}/match/over", json=ov, timeout=5)
        last_state = r.json()
        assert_test(f"US-02: Over {ov[chr(39)+'over_number'+chr(39)]} submitted OK", r.status_code == 200)

    if last_state:
        assert_test("US-03: Score present",          "score"                  in last_state)
        assert_test("US-03: Win probability returned","win_probability"         in last_state)
        assert_test("US-04: Bowler recs returned",   "bowler_recommendations"  in last_state)
        assert_test("US-04: At least 1 bowler rec",  len(last_state.get("bowler_recommendations",[])) > 0)
        assert_test("US-05: Batter recs returned",   "batter_recommendations"  in last_state)
        assert_test("US-06: Alerts returned",        "alerts"                  in last_state)
        assert_test("US-07: Striker returned",       "striker"                 in last_state)
        wp = last_state.get("win_probability", -1)
        assert_test("US-09: Win prob 0-100",         0 <= wp <= 100, f"Got: {wp}")
        recs = last_state.get("bowler_recommendations", [])
        if recs:
            assert_test("US-10: Last bowler not re-recommended",
                        not any(r["name"] == overs_data[-1]["bowler"] for r in recs))

    r2 = requests.get(f"{BASE_URL}/players/profile/Rohit Sharma", timeout=5)
    assert_test("US-12: Player profile endpoint", r2.status_code == 200)
    r3 = requests.get(f"{BASE_URL}/players/top-batters?n=5", timeout=5)
    assert_test("US-13: Top batters endpoint", r3.status_code == 200)
    r4 = requests.get(f"{BASE_URL}/players/top-bowlers?n=5", timeout=5)
    assert_test("US-14: Top bowlers endpoint", r4.status_code == 200)

except Exception as e:
    assert_test("API reachable", False, str(e))
    print("Start server with: uvicorn app:app --reload")

print(f"Results: {passed} passed | {failed} failed")

[XX] API reachable
     -> HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /match/setup (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x1047ee650>: Failed to establish a new connection: [Errno 61] Connection refused'))
Start server with: uvicorn app:app --reload
Results: 0 passed | 1 failed


## 2. Full Match Simulation (End-to-End)

In [3]:
import numpy as np
np.random.seed(2024)

def simulate_full_match():
    requests.post(f"{BASE_URL}/match/setup", json={
        "team_a":"MI","team_b":"CSK",
        "team_a_players":TEAM_A,"team_b_players":TEAM_B,
        "batting_first":"A","venue":"Wankhede Stadium"}, timeout=5)

    bowlers = ["Deepak Chahar","Yuzvendra Chahal","Imran Tahir",
               "Shardul Thakur","Ravindra Jadeja","Moeen Ali"]
    bowler_overs = {b:0 for b in bowlers}
    last_bowler = None; win_probs = []

    print(f"{'Over':<6}{'Score':<10}{'CRR':<7}{'WP%':<8}{'Next Bowler Rec':<25}{'Alert'}")
    print("-"*80)

    for over in range(1, 21):
        available = [b for b in bowlers if b != last_bowler and bowler_overs.get(b,0) < 4]
        bowler = available[0] if available else bowlers[0]
        bowler_overs[bowler] += 1; last_bowler = bowler
        balls = [str(np.random.choice([0,1,2,4,6,"W"],p=[0.28,0.28,0.15,0.13,0.08,0.08])) for _ in range(6)]
        resp = requests.post(f"{BASE_URL}/match/over",
                             json={"over_number":over,"balls":balls,"bowler":bowler}, timeout=5)
        if resp.status_code != 200: print(f"Over {over} FAILED: {resp.status_code}"); continue
        s = resp.json()
        wp = s.get("win_probability",0); win_probs.append(wp)
        crr = s.get("crr",0); score = s.get("score","—")
        rec = s.get("bowler_recommendations",[{}])[0].get("name","—")[:22]
        alert = s.get("alerts",[{}])[0].get("title","—")[:20] if s.get("alerts") else "—"
        print(f"{over:<6}{score:<10}{crr:<7.2f}{wp:<8.1f}{rec:<25}{alert}")

    print("="*80)
    if win_probs:
        print(f"Win probability range: {min(win_probs):.1f}% - {max(win_probs):.1f}%")
    print("All 20 overs processed.")

try:
    simulate_full_match()
except Exception as e:
    print(f"Error: {e} — start API first.")

Error: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /match/setup (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x1051da150>: Failed to establish a new connection: [Errno 61] Connection refused')) — start API first.


## 3. Deployment Guide

In [4]:
print("""
====================================================
DEPLOYMENT GUIDE
====================================================

LOCAL DEV:
  pip install -r requirements.txt
  uvicorn app:app --reload --port 8000
  Open: http://localhost:8000/ipl_dashboard.html

RENDER.COM (free):
  Build: pip install -r requirements.txt
  Start: uvicorn app:app --host 0.0.0.0 --port 
  Update API_BASE in dashboard HTML to render URL

DOCKER:
  docker build -t ipl-dashboard .
  docker run -p 8000:8000 ipl-dashboard
====================================================""")

with open("Dockerfile","w") as f:
    f.write("FROM python:3.11-slim
")
    f.write("WORKDIR /app
")
    f.write("COPY requirements.txt .
")
    f.write("RUN pip install -r requirements.txt
")
    f.write("COPY . .
")
    f.write("EXPOSE 8000
")
    f.write("CMD ["uvicorn","app:app","--host","0.0.0.0","--port","8000"]
")
print("Dockerfile written.")

SyntaxError: unterminated string literal (detected at line 22) (5933764.py, line 22)

## Sprint Summary

| Sprint | Notebook | Key Output |
|---|---|---|
| 1 Foundation | 01_Data_Pipeline.ipynb | master_players.csv, feature dataset |
| 2 Infrastructure | 03_Backend_API.ipynb | app.py — FastAPI + WebSocket |
| 3 Intelligence | 02_ML_Models.ipynb | Win Prob, Impact Score, Recommender |
| 4 Visualization | 04_Dashboard_Integration.ipynb | API-connected dashboard JS |
| 5 Launch | 05_Testing_Deployment.ipynb | 14 user story tests, full match sim |

**All 7 Jira Epics covered:** Data Pipeline · ML Models · Backend · UI · Decision Engine · Testing · Deployment